# Neural Machine Translation (English to Vietnamese)

This notebook implements a Seq2Seq model with Bahdanau Attention for English-Vietnamese translation.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.model_selection import train_test_split
import unicodedata
import re
import numpy as np
import os
import io
import time
import tarfile
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

## 1. Download and Prepare Data

In [ ]:
OUT_DIR = "data_iwslt15"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
    SITE_PREFIX="https://github.com/stefan-it/nmt-en-vi/raw/master/data"
    print("Downloading dataset...")
    !curl -L -o "{OUT_DIR}/train.tgz" "{SITE_PREFIX}/train-en-vi.tgz"
    !curl -L -o "{OUT_DIR}/dev.tgz" "{SITE_PREFIX}/dev-2012-en-vi.tgz"
    !curl -L -o "{OUT_DIR}/test.tgz" "{SITE_PREFIX}/test-2013-en-vi.tgz"
    
    print("Extracting dataset...")
    with tarfile.open(f"{OUT_DIR}/train.tgz", "r:gz") as tar:
        tar.extractall(path=OUT_DIR)
    with tarfile.open(f"{OUT_DIR}/dev.tgz", "r:gz") as tar:
        tar.extractall(path=OUT_DIR)
    with tarfile.open(f"{OUT_DIR}/test.tgz", "r:gz") as tar:
        tar.extractall(path=OUT_DIR)

print("Files in OUT_DIR:", os.listdir(OUT_DIR))

In [ ]:
def preprocess_sentence(w):
    w = w.lower().strip()
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)
    w = w.strip()
    w = '<start> ' + w + ' <end>'
    return w

def load_data(source_file, target_file, number_of_examples=None):
    if not os.path.exists(source_file) or not os.path.exists(target_file):
        return [], []
    source_sents = open(source_file, "r", encoding='utf-8').readlines()
    target_sents = open(target_file, "r", encoding='utf-8').readlines()
    
    min_len = min(len(source_sents), len(target_sents))
    source_sents = source_sents[:min_len]
    target_sents = target_sents[:min_len]

    max_len = 50
    source_data, target_data = [], []
    
    count = 0
    for source_sentence, target_sentence in zip(source_sents, target_sents):
        if number_of_examples is not None and count >= number_of_examples:
            break
        src = source_sentence.strip()
        trg = target_sentence.strip()
        if len(src.split()) > max_len or len(trg.split()) > max_len:
            continue
        source_data.append(preprocess_sentence(src))
        target_data.append(preprocess_sentence(trg))
        count += 1
    return source_data, target_data

def tokenize(sentences):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
    tokenizer.fit_on_texts(sentences)
    tensor = tokenizer.texts_to_sequences(sentences)
    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')
    return tensor, tokenizer

In [ ]:
num_examples = 50000 # Set to a higher value for better quality
input_tensor, target_tensor, inp_lang, targ_lang = None, None, None, None

def create_dataset(path_en, path_vi, num_examples):
    src_data, trg_data = load_data(path_en, path_vi, num_examples)
    if not src_data:
        return None, None, None, None
    input_tensor, inp_lang = tokenize(src_data)
    target_tensor, targ_lang = tokenize(trg_data)
    return input_tensor, target_tensor, inp_lang, targ_lang

input_tensor, target_tensor, inp_lang, targ_lang = create_dataset(f"{OUT_DIR}/train.en", f"{OUT_DIR}/train.vi", num_examples)
if input_tensor is None:
    input_tensor, target_tensor, inp_lang, targ_lang = create_dataset(f"{OUT_DIR}/train-en-vi.en", f"{OUT_DIR}/train-en-vi.vi", num_examples)

max_length_targ, max_length_inp = target_tensor.shape[1], input_tensor.shape[1]
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor, test_size=0.2)

BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64
steps_per_epoch = len(input_tensor_train)//BATCH_SIZE
embedding_dim = 256
units = 1024
vocab_inp_size = len(inp_lang.word_index) + 1
vocab_tar_size = len(targ_lang.word_index) + 1

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

## 2. Define Model

In [ ]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state = hidden)
        return output, state

    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.enc_units))

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.dec_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state = self.gru(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state, attention_weights

In [ ]:
encoder = Encoder(vocab_inp_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

## 3. Training

In [ ]:
@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        dec_hidden = enc_hidden
        dec_input = tf.expand_dims([targ_lang.word_index['<start>']] * BATCH_SIZE, 1)
        for t in range(1, targ.shape[1]):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1)
    batch_loss = (loss / int(targ.shape[1]))
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

EPOCHS = 10
for epoch in range(EPOCHS):
    start = time.time()
    enc_hidden = encoder.initialize_hidden_state()
    total_loss = 0
    for (batch, (inp, targ)) in enumerate(dataset.take(steps_per_epoch)):
        batch_loss = train_step(inp, targ, enc_hidden)
        total_loss += batch_loss
        if batch % 100 == 0:
            print(f'Epoch {epoch + 1} Batch {batch} Loss {batch_loss.numpy():.4f}')
    print(f'Epoch {epoch + 1} Loss {total_loss/steps_per_epoch:.4f}')
    print(f'Time taken for 1 epoch {time.time() - start:.2f} sec\n')

## 4. Evaluation and Prediction

In [ ]:
def evaluate(sentence):
    attention_plot = np.zeros((max_length_targ, max_length_inp))
    sentence = preprocess_sentence(sentence)
    inputs = [inp_lang.word_index.get(i, 0) for i in sentence.split(' ')]
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs], maxlen=max_length_inp, padding='post')
    inputs = tf.convert_to_tensor(inputs)
    result = ''
    hidden = [tf.zeros((1, units))]
    enc_out, enc_hidden = encoder(inputs, hidden)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([targ_lang.word_index['<start>']], 0)
    for t in range(max_length_targ):
        predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_out)
        predicted_id = tf.argmax(predictions[0]).numpy()
        word = targ_lang.index_word.get(predicted_id, '')
        if word == '<end>':
            return result.strip(), sentence, attention_plot
        result += word + ' '
        dec_input = tf.expand_dims([predicted_id], 0)
    return result.strip(), sentence, attention_plot

def translate(sentence):
    result, sentence, attention_plot = evaluate(sentence)
    print(f'Input: {sentence}')
    print(f'Predicted translation: {result}')

translate("how are you ?")
translate("i love natural language processing .")